In [1]:
import pandas as pd
from dotenv import load_dotenv
from bertopic import BERTopic

load_dotenv()
ACTIVE_BACKEND = "GPU"
TEST_DOC_LIMIT = 100
PARQUET_PATH = r"C:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\ADS_Pipeline\runs\run_20260310_180458_ADS_Curation_Run\data\publications_disambiguated.parquet"

df = pd.read_parquet(PARQUET_PATH)
docs = df["full_text"].dropna().astype(str).str.strip()
docs = docs[docs != ""].tolist()[:TEST_DOC_LIMIT] if TEST_DOC_LIMIT else docs[docs != ""].tolist()

PHYSICS_PROMPT = (
    "You are an experienced researcher in gravitational physics, astrophysics, and cosmology. "
    "You are labeling research topic clusters based on scientific abstracts.\n\n"
    "Documents: [DOCUMENTS]\n"
    "Keywords: [KEYWORDS]\n\n"
    "Task:\n"
    "- Generate EXACTLY ONE topic label.\n"
    "- The label MUST contain between 4 and 7 words (inclusive).\n"
    "- The label should read like a review article or textbook chapter title.\n\n"
    "Guidelines:\n"
    "- Use standard scientific terminology from the field.\n"
    "- Be specific about the physical phenomenon or method.\n"
    "- Avoid generic terms like 'studies', 'analysis', or 'research'.\n\n"
    "Output format (single line):\n"
    "topic: <label>\n\n"
    "Do NOT write anything else."
)

print(f"Loaded {len(docs)} documents.")

Loaded 100 documents.


In [ ]:
representation_model = None

if ACTIVE_BACKEND == "API":
    from bertopic.representation import LiteLLM

    model_name = "openrouter/google/gemini-3.1-flash-lite-preview"
    print(f"API backend: {model_name}")
    representation_model = LiteLLM(model=model_name, prompt=PHYSICS_PROMPT)

elif ACTIVE_BACKEND == "CPU":
    from bertopic.representation import LlamaCPP
    from huggingface_hub import hf_hub_download
    from llama_cpp import Llama

    model_path = hf_hub_download(
        repo_id="Qwen/Qwen2.5-0.5B-Instruct-GGUF", # "google/gemma-3-1b-it-qat-q4_0-gguf"
        filename="qwen2.5-0.5b-instruct-q4_k_m.gguf", # "gemma-3-1b-it-q4_0.gguf"
    )
    print("CPU backend: qwen2.5-0.5b-instruct-q4_k_m.gguf")
    llm = Llama(model_path=model_path, n_gpu_layers=0, n_ctx=16384, stop=["\n"], verbose=False)
    representation_model = LlamaCPP(llm, prompt=PHYSICS_PROMPT, pipeline_kwargs={"max_tokens": 50})

elif ACTIVE_BACKEND == "GPU":
    from bertopic.representation import TextGeneration
    from transformers import pipeline

    model_name = "Qwen/Qwen2.5-0.5B-Instruct" # "google/gemma-3-1b-it"
    print(f"GPU backend: {model_name}")
    generator = pipeline("text-generation", model=model_name, device_map="auto", dtype="auto")
    generation_config = generator.model.generation_config
    generation_config.max_length = None
    generation_config.max_new_tokens = 50
    generation_config.do_sample = False
    generation_config.num_return_sequences = 1
    generation_config.temperature = None
    generation_config.top_p = None
    generation_config.top_k = None
    representation_model = TextGeneration(
        generator,
        prompt=PHYSICS_PROMPT,
        pipeline_kwargs={"generation_config": generation_config},
    )

else:
    raise ValueError(f"Invalid ACTIVE_BACKEND: {ACTIVE_BACKEND}")

GPU backend: Qwen/Qwen2.5-0.5B-Instruct


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [3]:
print("Running BERTopic...")
topic_model = BERTopic(representation_model=representation_model, verbose=True)
topics, probs = topic_model.fit_transform(docs)
print(topic_model.get_topic_info().head(10))

2026-03-11 15:48:08,620 - BERTopic - Embedding - Transforming documents to embeddings.


Running BERTopic...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

2026-03-11 15:48:13,700 - BERTopic - Embedding - Completed ✓
2026-03-11 15:48:13,702 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-11 15:48:25,875 - BERTopic - Dimensionality - Completed ✓
2026-03-11 15:48:25,877 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-11 15:48:25,885 - BERTopic - Cluster - Completed ✓
2026-03-11 15:48:25,892 - BERTopic - Representation - Fine-tuning topics using representation models.
100%|██████████| 3/3 [02:31<00:00, 50.66s/it]
2026-03-11 15:50:57,940 - BERTopic - Representation - Completed ✓


   Topic  Count                                               Name  \
0     -1     66  -1_ This task requires you to generate a new t...   
1      0     23  0_ This task requires you to generate a topic ...   
2      1     11  1_ This task is purely for generating a random...   

                                      Representation  \
0  [ This task requires you to generate a new top...   
1  [ This task requires you to generate a topic l...   
2  [ This task is purely for generating a random ...   

                                 Representative_Docs  
0  [The meaning of quantum gravity.. Should the g...  
1  [The telescopic principles of the theory of gr...  
2  [Boltzmann's cosmogony and the hierarchical st...  
